# Collect and clean: sponsor pages of League of Legends organizations, 2019–2026

**Michael Chow**

This notebook collects archived "Partners" and "Sponsors" pages from the [Internet Archive's Wayback Machine](https://web.archive.org/) and turns them into a tidy dataset of sponsors. It follows the methods of **Week 7: Archived Web Pages and the Wayback Machine** (`ch-07-archives.ipynb`) and **Week 6: Parsing static web pages**. Cells adapted from the class notebook say so in a comment.

| Step | What it does | Class method |
| --- | --- | --- |
| 1. Find the pages | Ask the CDX API which partner/sponsor URLs are archived for each organization | CDX Server API (Ch. 7) |
| 2. List and choose captures | List every capture of each page; pick one early-year and one late-year capture per year | CDX API → DataFrame (Ch. 7, exercise Steps 1–2) |
| 3. Download | Fetch each chosen capture with the `id_` suffix and cache it to disk | `fetch_raw_capture` (Ch. 7, exercise Step 4) |
| 4. Parse | Pull sponsor links and logos out of each page with BeautifulSoup | Parsing static pages (Ch. 6) |
| 5. Clean | Normalize names, remove duplicates, add hand-coded industries | pandas |
| 6. Export | Save the clean data, a repository sample and CSVs for charts | |

**Polite collection.** Every request uses `get_with_retry` from the Ch. 7 exercise: a User-Agent that identifies me, a pause before every request, and waiting and retrying on 429 or 5xx responses. Every response is cached, so rerunning the notebook never downloads anything twice. Only the Internet Archive is contacted, never the teams' live websites.

**Why the CDX API rather than the Availability API.** The Availability API (`archive.org/wayback/available`) returns one capture per request, so choosing 16 dates per page would take 16 requests. The CDX API lists every capture of a page in a single request, which is far fewer requests for the Archive.

In [2]:
import json
import os
import re
import time
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse

import pandas as pd
import requests
from bs4 import BeautifulSoup

# Identify myself to the Archive (same convention as the class notebooks)
HEADERS = {"User-Agent": "EsportsSponsorshipStudy/1.0 (student class project; contact: mich3094@colorado.edu)"}
PAUSE = 3   # seconds to wait before every request ("Be respectful of the Archive")

YEARS = list(range(2019, 2027))   # 2019-2026

# region = where the organization's League team competes
ORGS = [
    {"org": "Team Liquid",   "region": "North America", "domains": ["teamliquid.com"]},
    {"org": "Cloud9",        "region": "North America", "domains": ["cloud9.gg"]},
    {"org": "FlyQuest",      "region": "North America", "domains": ["flyquest.gg"]},
    {"org": "G2 Esports",    "region": "Europe",        "domains": ["g2esports.com"]},
    {"org": "Fnatic",        "region": "Europe",        "domains": ["fnatic.com"]},
    {"org": "Team Vitality", "region": "Europe",        "domains": ["vitality.gg", "teamvitality.gg"]},
    {"org": "SK Gaming",     "region": "Europe",        "domains": ["sk-gaming.com"]},
    {"org": "T1",            "region": "Korea",         "domains": ["t1.gg"]},
    {"org": "Gen.G",         "region": "Korea",         "domains": ["geng.gg", "en.geng.gg"]},
    # Unconfirmed domains: Step 1 shows whether the Archive has anything for them.
    {"org": "Dplus KIA",     "region": "Korea",         "domains": ["dpluskia.com", "dplus-kia.com"]},
]

# Pages to add or drop by hand after looking at Step 1's output.
# Example: MANUAL_PAGES = {"T1": ["t1.gg/about/partners"]}
MANUAL_PAGES = {}
EXCLUDE_PAGES = []

DATA = Path("data")
CDX_DIR = DATA / "raw" / "cdx"
HTML_DIR = DATA / "raw" / "html"
INTERIM = DATA / "interim"
CLEAN = DATA / "clean"
SAMPLE = DATA / "sample"
for folder in [CDX_DIR, HTML_DIR, INTERIM, CLEAN, SAMPLE]:
    os.makedirs(folder, exist_ok=True)

ORG_INFO = pd.DataFrame(ORGS)


def slug(text):
    """Turn any text into a safe file name."""
    return re.sub(r"[^a-z0-9]+", "_", str(text).lower()).strip("_")

## Polite requests and the CDX API

`get_with_retry` is the function from the Ch. 7 exercise (Step 1). I added the pause before each request and a log of every request, so the README can report how many were made. `cdx_query` builds a DataFrame from the CDX response the same way as the exercise's Step 2, and caches the response like `get_wayback_content_cached` in the chapter.

In [4]:
request_log = []   # one entry per request: url, status, time


# Adapted from ch-07-archives.ipynb, exercise Step 1
def get_with_retry(url, params=None, headers=HEADERS, max_tries=5, wait=30, timeout=120):
    """GET a URL. Treat 429 (busy) and 5xx (overloaded) as outcomes: wait and retry."""
    for attempt in range(1, max_tries + 1):
        time.sleep(PAUSE)  # Be respectful of the Archive before every request
        try:
            response = requests.get(url, params=params, headers=headers, timeout=timeout)
        except requests.exceptions.RequestException as e:
            print(f"Attempt {attempt}: request failed ({e}). Waiting {wait * attempt}s...")
            request_log.append({"url": url, "status": "error", "time": datetime.now().isoformat()})
            time.sleep(wait * attempt)
            continue
        request_log.append({"url": url, "status": response.status_code, "time": datetime.now().isoformat()})

        if response.status_code == 429 or response.status_code >= 500:
            # Use the server's Retry-After if it gives one, otherwise back off longer each time
            try:
                delay = int(response.headers.get("Retry-After", wait * attempt))
            except ValueError:
                delay = wait * attempt
            print(f"Attempt {attempt}: got {response.status_code}. Waiting {delay}s and retrying...")
            time.sleep(delay)
            continue

        response.raise_for_status()  # Any other error (e.g. 404) is a real problem
        return response

    raise RuntimeError(f"Gave up on {url} after {max_tries} attempts")


CDX_URL = "https://web.archive.org/cdx/search/cdx"


def cdx_query(params, cache_path):
    """Query the CDX API and return the captures as a DataFrame, using a local cache.
    Adapted from ch-07-archives.ipynb (exercise Steps 1-2 and get_wayback_content_cached)."""
    if os.path.exists(cache_path):  # Already fetched, so skip the network
        with open(cache_path, "r") as f:
            data = json.load(f)
    else:
        try:
            response = get_with_retry(CDX_URL, params={**params, "output": "json"})
        except (RuntimeError, requests.exceptions.HTTPError) as e:
            print(f"  CDX query failed for {params['url']}: {e}")
            return pd.DataFrame()
        data = response.json() if response.text.strip() else []
        with open(cache_path, "w") as f:
            json.dump(data, f)

    if len(data) < 2:   # no captures
        return pd.DataFrame()
    # First row is the column headers, the rest is data
    return pd.DataFrame(data[1:], columns=data[0])

## Step 1: Find each organization's partner and sponsor pages

Each CDX query uses `matchType=prefix` to list every archived URL that *starts with* a likely path, such as `cloud9.gg/partner` or `vitality.gg/en/pages/partner`. `collapse=urlkey` returns each URL once. The notebook keeps URLs that look like an index page (`/partners`, `/en/partners`, `/pages/sponsors`...) rather than single news posts. **Check the table this step prints.** If a page is wrong or missing, fix it with `MANUAL_PAGES` / `EXCLUDE_PAGES` above and rerun.

In [6]:
PREFIXES = ["", "en/", "pages/", "en/pages/", "about/"]
WORDS = ["partner", "sponsor"]
INDEX_NAMES = {"partner", "partners", "sponsor", "sponsors", "partnerships", "our-partners",
               "partenaires", "nos-partenaires"}
SKIP_SEGMENTS = {"en", "fr", "de", "es", "ko", "kr", "pages", "about"}


def host_and_path(original):
    """'https://www.cloud9.gg/partners/?x=1' -> 'cloud9.gg/partners'"""
    parsed = urlparse(original if "://" in original else "http://" + original)
    host = parsed.netloc.lower().split(":")[0].removeprefix("www.")
    return host + parsed.path.rstrip("/")


def looks_like_index(original):
    path = urlparse(original if "://" in original else "http://" + original).path.lower()
    parts = [p for p in path.strip("/").split("/") if p and p not in SKIP_SEGMENTS]
    return len(parts) == 1 and parts[0] in INDEX_NAMES


found = []
for org in ORGS:
    print(org["org"])
    for domain in org["domains"]:
        for prefix in PREFIXES:
            for word in WORDS:
                query = f"{domain}/{prefix}{word}"
                params = {"url": query, "matchType": "prefix", "collapse": "urlkey",
                          "fl": "urlkey,original,timestamp,statuscode", "limit": "1000"}
                df = cdx_query(params, CDX_DIR / f"discover_{slug(query)}.json")
                if not df.empty:
                    df["org"] = org["org"]
                    found.append(df)

candidates = (pd.concat(found) if found else pd.DataFrame(columns=["org", "original", "timestamp"]))
candidates["page"] = candidates["original"].map(host_and_path)
candidates["first_capture"] = candidates["timestamp"].str[:8]
candidates["is_index"] = candidates["original"].map(looks_like_index)
candidates = (candidates.sort_values("first_capture")
              .drop_duplicates(["org", "page"])
              .sort_values(["org", "page"])[["org", "page", "first_capture", "is_index"]])
candidates.to_csv(INTERIM / "candidate_pages.csv", index=False)

pages = candidates[candidates["is_index"] & ~candidates["page"].isin(EXCLUDE_PAGES)][["org", "page"]]
manual = pd.DataFrame([{"org": o, "page": host_and_path(p)} for o, ps in MANUAL_PAGES.items() for p in ps],
                      columns=["org", "page"])
pages = pd.concat([pages, manual]).drop_duplicates().reset_index(drop=True)

print(f"\n{len(candidates)} archived URLs matched; {len(pages)} kept as partner/sponsor index pages:\n")
print(pages.to_string(index=False))
missing_orgs = sorted(set(ORG_INFO["org"]) - set(pages["org"]))
if missing_orgs:
    print("\nNo partner page found for:", ", ".join(missing_orgs),
          "\nLook in data/interim/candidate_pages.csv or the Wayback Machine, then add pages to MANUAL_PAGES.")

Team Liquid
Cloud9
FlyQuest
G2 Esports
Fnatic
Team Vitality
SK Gaming
T1
Gen.G
Dplus KIA

317 archived URLs matched; 24 kept as partner/sponsor index pages:

          org                          page
       Cloud9      cloud9.gg/pages/partners
       Cloud9            cloud9.gg/partners
       Cloud9            cloud9.gg/sponsors
     FlyQuest    flyquest.gg/pages/partners
     FlyQuest          flyquest.gg/partners
       Fnatic           fnatic.com/partners
       Fnatic            fnatic.com/sponsor
       Fnatic           fnatic.com/sponsors
   G2 Esports  g2esports.com/about/partners
   G2 Esports  g2esports.com/pages/partners
   G2 Esports        g2esports.com/partners
        Gen.G     en.geng.gg/pages/partners
        Gen.G        geng.gg/pages/partners
        Gen.G              geng.gg/partners
    SK Gaming        sk-gaming.com/partners
    SK Gaming         sk-gaming.com/sponsor
           T1                t1.gg/partners
  Team Liquid       teamliquid.com/partners
  Team

## Step 2: List captures and choose two per organization per year

For every page from Step 1, one CDX query lists all successful HTML captures from 2019 to 2026. `filter` keeps status 200 and `collapse=timestamp:8` keeps at most one a day. For each organization and year the notebook picks:

- **early:** the first capture in January–February. If there is none, the closest capture from March–May.
- **late:** the last capture in November–December. If there is none, the closest capture from August–October.

`in_target_window` records whether a fallback month was used. A year with nothing at all is kept as a `missing` row, so gaps stay visible. The chosen captures are also saved as JSON, like `fb_tos_history.json` in the chapter.

In [8]:
all_captures = []
for page_row in pages.itertuples():
    params = {"url": page_row.page, "from": "2019", "to": "2026",
              "filter": ["statuscode:200", "mimetype:text/html"],
              "collapse": "timestamp:8", "fl": "timestamp,original,statuscode,mimetype,length"}
    df = cdx_query(params, CDX_DIR / f"captures_{slug(page_row.page)}.json")
    if not df.empty:
        df["org"] = page_row.org
        df["page"] = page_row.page
        all_captures.append(df)

all_captures = (pd.concat(all_captures, ignore_index=True) if all_captures
                else pd.DataFrame(columns=["org", "page", "timestamp", "original", "length"]))
# Real datetimes and numeric lengths, as in the Ch. 7 exercise (Step 2)
all_captures["captured_at"] = pd.to_datetime(all_captures["timestamp"], format="%Y%m%d%H%M%S")
all_captures["length"] = pd.to_numeric(all_captures["length"], errors="coerce")
print(f"{len(all_captures)} daily captures found across {all_captures['page'].nunique()} pages")
if len(all_captures):
    print(f"Date range: {all_captures['captured_at'].min()} to {all_captures['captured_at'].max()}")

PERIODS = {
    # period: (months allowed, target months, target date as (month, day))
    "early": ([1, 2, 3, 4, 5], [1, 2], (1, 1)),
    "late": ([8, 9, 10, 11, 12], [11, 12], (12, 31)),
}

selected = []
for org in ORG_INFO["org"]:
    org_caps = all_captures[all_captures["org"] == org]
    for year in YEARS:
        for period, (allowed, ideal, (month, day)) in PERIODS.items():
            pool = org_caps[(org_caps["captured_at"].dt.year == year)
                            & org_caps["captured_at"].dt.month.isin(allowed)].copy()
            base = {"org": org, "year": year, "period": period}
            if pool.empty:
                selected.append({**base, "status": "missing"})
                continue
            target = pd.Timestamp(year, month, day)
            pool["in_target_window"] = pool["captured_at"].dt.month.isin(ideal)
            pool["days_from_target"] = (pool["captured_at"] - target).dt.days.abs()
            best = pool.sort_values(["in_target_window", "days_from_target"],
                                    ascending=[False, True]).iloc[0]
            selected.append({**base, "status": "selected", "timestamp": best["timestamp"],
                             "captured_at": best["captured_at"], "page_url": best["original"],
                             "in_target_window": bool(best["in_target_window"]),
                             "days_from_target": int(best["days_from_target"])})

captures = pd.DataFrame(selected)
captures = captures.merge(ORG_INFO[["org", "region"]], on="org", how="left")
captures["snapshot_url"] = [
    f"https://web.archive.org/web/{ts}/{url}" if isinstance(ts, str) else ""
    for ts, url in zip(captures.get("timestamp", [None] * len(captures)),
                       captures.get("page_url", [None] * len(captures)))
]

# Save the chosen captures as JSON for reproducibility (Ch. 7, "fb_tos_history.json")
with open(INTERIM / "captures_selected.json", "w") as f:
    json.dump(captures.drop(columns="captured_at", errors="ignore").to_dict("records"), f, indent=2, default=str)

print(f"{(captures['status'] == 'selected').sum()} captures selected, "
      f"{(captures['status'] == 'missing').sum()} organization-year-periods missing\n")
print("Captures selected per organization and year (2 = early and late both found):")
print(captures[captures["status"] == "selected"]
      .pivot_table(index="org", columns="year", values="period", aggfunc="count", fill_value=0)
      .reindex(index=ORG_INFO["org"], columns=YEARS, fill_value=0).to_string())

531 daily captures found across 18 pages
Date range: 2019-01-01 19:35:06 to 2026-10-20 02:14:08
112 captures selected, 48 organization-year-periods missing

Captures selected per organization and year (2 = early and late both found):
year           2019  2020  2021  2022  2023  2024  2025  2026
org                                                          
Team Liquid       2     2     2     2     2     2     2     2
Cloud9            2     2     2     2     2     2     2     2
FlyQuest          1     1     0     2     2     2     2     2
G2 Esports        2     2     1     2     2     2     2     2
Fnatic            2     0     2     2     2     2     2     2
Team Vitality     1     1     2     2     2     2     2     2
SK Gaming         0     0     0     1     2     2     2     2
T1                0     1     2     1     2     2     2     0
Gen.G             0     0     1     2     2     1     0     0
Dplus KIA         0     0     0     0     0     0     0     0


## Step 3: Download the selected captures

`fetch_raw_capture` comes from the Ch. 7 exercise (Step 4). Adding `id_` after the timestamp returns the page exactly as it was archived, with no Wayback toolbar and no rewritten links. Raw bytes are saved to `data/raw/html/<org>/<timestamp>.html` so old encodings survive, and files already on disk are skipped.

In [10]:
def html_path(org, timestamp):
    return HTML_DIR / slug(org) / f"{timestamp}.html"


# Adapted from ch-07-archives.ipynb, exercise Step 4
def fetch_raw_capture(org, timestamp, original):
    """Fetch the original (id_) capture for one chosen capture, using a local cache."""
    cache_path = html_path(org, timestamp)

    if os.path.exists(cache_path):  # Already fetched, so skip the network
        return "cached"

    # id_ after the timestamp = the raw capture, with no toolbar and no rewritten links
    snapshot_url = f"https://web.archive.org/web/{timestamp}id_/{original}"
    try:
        response = get_with_retry(snapshot_url)
    except (RuntimeError, requests.exceptions.HTTPError) as e:
        print(f"  Could not fetch {snapshot_url}: {e}")
        return "failed"

    os.makedirs(cache_path.parent, exist_ok=True)
    with open(cache_path, "wb") as f:  # Save raw bytes so old encodings survive
        f.write(response.content)
    return "downloaded"


to_get = captures[captures["status"] == "selected"]
download_status = []
for i, row in enumerate(to_get.itertuples(), start=1):
    if not os.path.exists(html_path(row.org, row.timestamp)):
        print(f"[{i}/{len(to_get)}] {row.org} {row.year} {row.period}")
    download_status.append(fetch_raw_capture(row.org, row.timestamp, row.page_url))

captures.loc[to_get.index, "download"] = download_status
print(captures.loc[to_get.index, "download"].value_counts().to_string())

download
cached    112


## Step 4: Parse sponsor mentions from each page

Sponsors usually appear as logos that link to the sponsor's own website. The notebook records two kinds of **mentions**:

- **link:** every link to an outside website, with the logo's `alt` text if the link wraps an image. Links to the organization's own sites, social media, app stores, shops and other platforms are skipped (`SKIP_DOMAINS`).
- **image:** logo images inside an element whose `class` or `id` contains "partner" or "sponsor", when the image has no link.

This step keeps every mention, including some that are not sponsors. Step 5 decides what counts.

In [12]:
SKIP_DOMAINS = {
    # social media and video
    "facebook.com", "twitter.com", "x.com", "instagram.com", "youtube.com", "youtu.be", "twitch.tv",
    "tiktok.com", "discord.gg", "discord.com", "reddit.com", "linkedin.com", "snapchat.com",
    "weibo.com", "vk.com", "threads.net", "bsky.app", "kick.com", "naver.com", "afreecatv.com",
    "sooplive.co.kr", "chzzk.naver.com", "spotify.com",
    # app stores, web infrastructure, archive
    "apple.com", "google.com", "goo.gl", "bit.ly", "linktr.ee", "archive.org", "shopify.com",
    "myshopify.com", "cloudfront.net", "googleapis.com", "gstatic.com", "wp.com", "wordpress.com",
    "wordpress.org", "typeform.com", "mailchimp.com", "list-manage.com", "onetrust.com",
    "cookiebot.com", "w3.org", "schema.org", "jquery.com", "squarespace.com", "wix.com",
    # leagues and esports sites (not sponsors)
    "lolesports.com", "riotgames.com", "leagueoflegends.com", "liquipedia.net", "hltv.org",
}
SECOND_LEVEL = {"co", "com", "org", "net", "ac", "or", "ne", "go"}


def registered_domain(host):
    """'shop.redbull.co.uk' -> 'redbull.co.uk'; 'www.redbull.com' -> 'redbull.com'"""
    parts = host.lower().split(":")[0].strip(".").split(".")
    if len(parts) >= 3 and parts[-2] in SECOND_LEVEL and len(parts[-1]) == 2:
        return ".".join(parts[-3:])
    return ".".join(parts[-2:])


def unwrap_wayback(href):
    """Turn a rewritten Wayback link back into the original URL."""
    match = re.search(r"web\.archive\.org/web/\d+[a-z_]*/(.+)$", href)
    return match.group(1) if match else href


def is_skipped(domain, org_domains):
    return domain in org_domains or any(domain == d or domain.endswith("." + d) for d in SKIP_DOMAINS)


def parse_page(html, org_domains):
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style"]):
        tag.decompose()
    mentions = []

    for link in soup.find_all("a", href=True):
        href = unwrap_wayback(link["href"].strip())
        if href.startswith("//"):
            href = "https:" + href
        if not href.startswith(("http://", "https://")):
            continue                      # internal or mailto/javascript links
        host = urlparse(href).netloc
        if not host:
            continue
        domain = registered_domain(host)
        if is_skipped(domain, org_domains):
            continue
        image = link.find("img")
        mentions.append({
            "mention_type": "link",
            "sponsor_domain": domain,
            "img_alt": (image.get("alt") or image.get("title") or "").strip() if image else "",
            "img_src": (image.get("src") or image.get("data-src") or "") if image else "",
            "link_text": link.get_text(" ", strip=True)[:120],
            "href": href[:300],
        })

    sections = (soup.find_all(class_=re.compile("partner|sponsor", re.I))
                + soup.find_all(id=re.compile("partner|sponsor", re.I)))
    seen = set()
    for section in sections:
        for image in section.find_all("img"):
            if id(image) in seen or image.find_parent("a", href=True):
                continue
            seen.add(id(image))
            mentions.append({
                "mention_type": "image",
                "sponsor_domain": "",
                "img_alt": (image.get("alt") or image.get("title") or "").strip(),
                "img_src": image.get("src") or image.get("data-src") or "",
                "link_text": "",
                "href": "",
            })
    return mentions


mention_rows = []
pages_parsed = []
for row in captures[captures["status"] == "selected"].itertuples():
    path = html_path(row.org, row.timestamp)
    if not path.exists():
        pages_parsed.append(None)
        continue
    org_domains = {registered_domain(d) for d in ORG_INFO.loc[ORG_INFO["org"] == row.org, "domains"].iloc[0]}
    mentions = parse_page(path.read_bytes(), org_domains)
    pages_parsed.append(len(mentions))
    for m in mentions:
        mention_rows.append({"org": row.org, "year": row.year, "period": row.period,
                             "timestamp": row.timestamp, **m})

captures.loc[captures["status"] == "selected", "n_mentions"] = pages_parsed
mentions = pd.DataFrame(mention_rows, columns=["org", "year", "period", "timestamp", "mention_type",
                                               "sponsor_domain", "img_alt", "img_src", "link_text", "href"])
mentions.to_csv(INTERIM / "sponsor_mentions.csv", index=False)
print(f"{len(mentions)} raw mentions from {captures['n_mentions'].notna().sum()} pages")
print("Pages with zero mentions (possibly built with JavaScript; may need Playwright):",
      int((captures["n_mentions"] == 0).sum()))
mentions.head(10)

2053 raw mentions from 112 pages
Pages with zero mentions (possibly built with JavaScript; may need Playwright): 8


,org,year,period,timestamp,mention_type,sponsor_domain,img_alt,img_src,link_text,href
0,Team Liquid,2019,early,20190325124759,link,alienware.com,,,,http://www.alienware.com/
1,Team Liquid,2019,early,20190325124759,link,alienware.com,,,alienware.com,http://www.alienware.com/
2,Team Liquid,2019,early,20190325124759,link,monsterenergy.com,,,,https://www.monsterenergy.com/us/en/products/m...
3,Team Liquid,2019,early,20190325124759,link,monsterenergy.com,,,monsterenergy.com,https://www.monsterenergy.com/us/en/products/m...
4,Team Liquid,2019,early,20190325124759,link,monsterenergy.com,,,www.monsterenergy.com,https://www.monsterenergy.com/
5,Team Liquid,2019,early,20190325124759,link,tl.gg,,,,https://tl.gg/SAP
6,Team Liquid,2019,early,20190325124759,link,tl.gg,,,sap.com,https://tl.gg/SAP
7,Team Liquid,2019,early,20190325124759,link,tl.gg,,,,https://tl.gg/honda
8,Team Liquid,2019,early,20190325124759,link,honda.com,,,honda.com,https://www.honda.com/
9,Team Liquid,2019,early,20190325124759,link,jerseymikes.com,,,,https://www.jerseymikes.com/


## Step 5: Clean

Cleaning turns raw mentions into one row per **organization × year × period × sponsor**:

1. **Name each mention.** Every mention gets a `sponsor_key`. For links it's the website's name (`redbull.com` → `redbull`). For unlinked logos it's the cleaned `alt` text or image file name. Words like "logo", "partner" or "white" and file extensions are stripped.
2. **Merge spellings.** `data/brand_aliases.csv` maps extra keys onto one brand (for example `redbullgaming` → `redbull`) and sets display names. You edit this file by hand.
3. **Remove duplicates.** The same sponsor linked three times on one page counts once.
4. **Add industries.** `data/industry_codes.csv` holds the hand coding: one row per sponsor with an `industry`. New sponsors are appended with an empty `industry` each time this cell runs. Fill them in and rerun. Code anything that isn't really a sponsor (a ticket site, a charity link, a news article) as `not_a_sponsor`, and it's dropped.

In [14]:
INDUSTRIES = ["crypto_web3", "betting", "energy_drink_food", "pc_hardware_peripherals",
              "telecom_tech", "apparel", "automotive", "finance", "other", "not_a_sponsor"]
GENERIC_WORDS = {"logo", "logos", "partner", "partners", "sponsor", "sponsors", "official", "image",
                 "img", "icon", "white", "black", "color", "colour", "mono", "dark", "light", "new",
                 "png", "svg", "jpg", "jpeg", "webp", "gif", "the", "wordmark", "lockup", "horizontal", "vertical"}


def clean_text(text):
    """'RedBull_Logo-white.png' -> 'RedBull'"""
    text = re.sub(r"\.(png|svg|jpe?g|webp|gif)(\?.*)?$", "", str(text).strip(), flags=re.I)
    text = re.sub(r"[_\-]+", " ", text)
    words = [w for w in text.split()
             if w.lower().strip(".,:;!|()") not in GENERIC_WORDS
             and not re.fullmatch(r"\d+x\d+", w.lower())]   # drop image sizes like 225x92
    if words:   # 'Secretlab1225X71' -> 'Secretlab'
        words[-1] = re.sub(r"\d{3,}x\d+$", "", words[-1], flags=re.I) or words[-1]
    if len(words) > 1 and words[-1].isdigit():   # 'REWE 1' -> 'REWE' (but keep '7 Eleven')
        words = words[:-1]
    return " ".join(words).strip(" .,:;|")


def make_key(text):
    return re.sub(r"[^a-z0-9]", "", str(text).lower())


def domain_label(domain):
    """'redbull.co.uk' -> 'redbull'"""
    return domain.split(".")[0] if domain else ""


def mention_key(row):
    if row["sponsor_domain"]:
        return make_key(domain_label(row["sponsor_domain"]))
    if row["img_alt"]:
        return make_key(clean_text(row["img_alt"]))
    file_stem = Path(urlparse(row["img_src"]).path).name
    return make_key(clean_text(file_stem))


m = mentions.fillna("").copy()
# Name as written on the page: the logo's alt text (link text is often "Learn more", so it's not used)
m["name_from_page"] = m["img_alt"].map(clean_text)
m["sponsor_key"] = m.apply(mention_key, axis=1)

# Brand aliases: key -> canonical key + display name
alias_file = DATA / "brand_aliases.csv"
if not alias_file.exists():
    pd.DataFrame(columns=["raw_key", "sponsor_key", "sponsor_name"]).to_csv(alias_file, index=False)
aliases = pd.read_csv(alias_file, dtype=str).fillna("")
alias_map = dict(zip(aliases["raw_key"], aliases["sponsor_key"]))
m["sponsor_key"] = m["sponsor_key"].map(lambda k: alias_map.get(k) or k)

# Drop empty keys, very short keys and the organizations' own names
org_keys = {make_key(o) for o in ORG_INFO["org"]}
before = len(m)
m = m[(m["sponsor_key"].str.len() >= 2) & ~m["sponsor_key"].isin(org_keys)]
print(f"Dropped {before - len(m)} mentions with no usable name")

# Display name: alias name if set, otherwise the most common name seen on pages, otherwise the domain
display = {}
for key, group in m.groupby("sponsor_key"):
    names = group.loc[group["name_from_page"].str.len().between(2, 40), "name_from_page"]
    domains = group.loc[group["sponsor_domain"] != "", "sponsor_domain"]
    display[key] = (names.mode().iloc[0] if not names.empty
                    else domains.mode().iloc[0] if not domains.empty else key.title())
display.update({k: n for k, n in zip(aliases["sponsor_key"], aliases["sponsor_name"]) if n})

# One row per org x year x period x sponsor
tidy = (m.groupby(["org", "year", "period", "timestamp", "sponsor_key"], as_index=False)
         .agg(sponsor_domain=("sponsor_domain", lambda s: next((d for d in s if d), "")),
              mention_types=("mention_type", lambda s: ",".join(sorted(set(s)))),
              n_mentions=("mention_type", "size")))
tidy["sponsor_name"] = tidy["sponsor_key"].map(display)

# Industry codes: keep existing hand codes, append new sponsors with a blank industry
codes_file = DATA / "industry_codes.csv"
codes = (pd.read_csv(codes_file, dtype=str).fillna("") if codes_file.exists()
         else pd.DataFrame(columns=["sponsor_key", "sponsor_name", "sponsor_domain", "industry", "notes"]))
new = (tidy[~tidy["sponsor_key"].isin(codes["sponsor_key"])]
       .groupby("sponsor_key", as_index=False)
       .agg(sponsor_name=("sponsor_name", "first"), sponsor_domain=("sponsor_domain", "first")))
new["industry"], new["notes"] = "", ""
codes = pd.concat([codes, new], ignore_index=True)
codes = codes[~codes["sponsor_key"].isin(alias_map) | (codes["industry"] != "")]  # merged-away spellings
codes["n_org_years"] = codes["sponsor_key"].map(tidy.groupby("sponsor_key")[["org", "year"]]
                                                .apply(lambda g: len(g.drop_duplicates())))
codes = codes.sort_values(["industry", "n_org_years"], ascending=[True, False])
codes.drop(columns="n_org_years").to_csv(codes_file, index=False)

bad = sorted(set(codes["industry"]) - set(INDUSTRIES) - {""})
if bad:
    print("Unknown industry labels in industry_codes.csv (fix spelling):", bad)

tidy = tidy.merge(codes[["sponsor_key", "industry"]], on="sponsor_key", how="left")
tidy["industry"] = tidy["industry"].fillna("").replace("", "uncoded")
dropped = (tidy["industry"] == "not_a_sponsor").sum()
tidy = tidy[tidy["industry"] != "not_a_sponsor"]

# Add capture details and region, order columns
tidy = tidy.merge(captures[["org", "year", "period", "region", "captured_at", "snapshot_url"]],
                  on=["org", "year", "period"], how="left")
tidy["capture_date"] = pd.to_datetime(tidy["captured_at"]).dt.date
tidy = tidy[["org", "region", "year", "period", "capture_date", "sponsor_key", "sponsor_name",
             "sponsor_domain", "industry", "mention_types", "n_mentions", "snapshot_url"]]
tidy = tidy.sort_values(["org", "year", "period", "sponsor_name"]).reset_index(drop=True)

tidy.to_csv(CLEAN / "sponsors_clean.csv", index=False)

captures_out = captures.drop(columns=["captured_at"], errors="ignore")
captures_out["capture_date"] = pd.to_datetime(captures["captured_at"]).dt.date
captures_out["n_sponsors"] = captures_out.set_index(["org", "year", "period"]).index.map(
    tidy.groupby(["org", "year", "period"]).size()).fillna(0).astype(int)
captures_out = captures_out.reindex(columns=["org", "region", "year", "period", "status", "capture_date", "timestamp",
                             "page_url", "in_target_window", "days_from_target", "download",
                             "n_mentions", "n_sponsors", "snapshot_url"])
captures_out.to_csv(CLEAN / "captures.csv", index=False)

print(f"Removed {dropped} rows coded not_a_sponsor")
print(f"Clean dataset: {len(tidy)} rows, {tidy['sponsor_key'].nunique()} distinct sponsors")
print(f"Sponsors still to code by hand in data/industry_codes.csv: {(codes['industry'] == '').sum()}")
tidy.head(15)

Dropped 55 mentions with no usable name
Removed 0 rows coded not_a_sponsor
Clean dataset: 1007 rows, 240 distinct sponsors
Sponsors still to code by hand in data/industry_codes.csv: 264


,org,region,year,period,capture_date,sponsor_key,sponsor_name,sponsor_domain,industry,mention_types,n_mentions,snapshot_url
0,Cloud9,North America,2019,early,2019-01-29,c9,Kia,c9.gg,uncoded,link,1,https://web.archive.org/web/20190129010426/htt...
1,Cloud9,North America,2019,early,2019-01-29,kingston,Kingston,kingston.com,uncoded,link,1,https://web.archive.org/web/20190129010426/htt...
2,Cloud9,North America,2019,early,2019-01-29,airforce,airforce.com,airforce.com,uncoded,link,1,https://web.archive.org/web/20190129010426/htt...
3,Cloud9,North America,2019,early,2019-01-29,hp,hp.com,hp.com,uncoded,link,1,https://web.archive.org/web/20190129010426/htt...
4,Cloud9,North America,2019,late,2019-12-17,att,AT&T,att.com,uncoded,link,1,https://web.archive.org/web/20191217193319/htt...
5,Cloud9,North America,2019,late,2019-12-17,hyperx,HyperX,hyperx.gg,uncoded,link,1,https://web.archive.org/web/20191217193319/htt...
6,Cloud9,North America,2019,late,2019-12-17,c9,Kia,c9.gg,uncoded,link,4,https://web.archive.org/web/20191217193319/htt...
7,Cloud9,North America,2019,late,2019-12-17,kingston,Kingston,kingston.com,uncoded,link,1,https://web.archive.org/web/20191217193319/htt...
8,Cloud9,North America,2019,late,2019-12-17,microsoft,Microsoft,microsoft.com,uncoded,link,2,https://web.archive.org/web/20191217193319/htt...
9,Cloud9,North America,2019,late,2019-12-17,puma,PUMA,,uncoded,image,1,https://web.archive.org/web/20191217193319/htt...


## Save the sample for the repository and check the result

The repository holds only `data/sample/`: 60 random rows of the clean dataset and the full list of captures (timestamps and URLs only). Everything under `data/raw/`, `data/interim/` and `data/clean/` stays out of the repository (see the README's Data section).

In [16]:
sample = tidy.sample(min(60, len(tidy)), random_state=42).sort_values(["org", "year", "period"])
sample.to_csv(SAMPLE / "sponsors_sample.csv", index=False)
captures_out.to_csv(SAMPLE / "captures.csv", index=False)

# Checks on the tidy dataset
assert not tidy.duplicated(["org", "year", "period", "sponsor_key"]).any(), "duplicate sponsor rows"
assert tidy["sponsor_key"].str.len().ge(2).all()
assert tidy["year"].between(min(YEARS), max(YEARS)).all()

print("Numbers for the README's Data section")
print("-------------------------------------")
print("Collected on:", datetime.now().strftime("%B %d, %Y"))
print("Requests made in this run:", len(request_log))
print("Organizations with at least one capture:", captures.loc[captures["status"] == "selected", "org"].nunique(),
      "of", len(ORG_INFO))
print("Captures selected:", (captures["status"] == "selected").sum(), "of", len(captures),
      f"({(captures['status'] == 'missing').sum()} missing)")
print("Captures in the target months:", int(captures["in_target_window"].fillna(False).astype(bool).sum()))
print("Raw sponsor mentions:", len(mentions))
print("Clean rows (org x year x period x sponsor):", len(tidy))
print("Distinct sponsors:", tidy["sponsor_key"].nunique())
print("Rows still uncoded:", (tidy["industry"] == "uncoded").sum())
print("\nClean rows per year:")
print(tidy.groupby("year").size().reindex(YEARS, fill_value=0).to_string())

Numbers for the README's Data section
-------------------------------------
Collected on: November 08, 2026
Requests made in this run: 0
Organizations with at least one capture: 9 of 10
Captures selected: 112 of 160 (48 missing)
Captures in the target months: 81
Raw sponsor mentions: 2053
Clean rows (org x year x period x sponsor): 1007
Distinct sponsors: 240
Rows still uncoded: 1007

Clean rows per year:
year
2019     70
2020     80
2021    111
2022    205
2023    175
2024    124
2025    121
2026    121


/var/folders/s9/4m_nwrps2z7616h9335l6by00000gn/T/ipykernel_67113/1835738021.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  print("Captures in the target months:", int(captures["in_target_window"].fillna(False).astype(bool).sum()))


## Step 6: Export CSVs for charts (Power BI, Tableau or Excel)

Charting tools work best with **long** tables, one row per observation, and with readable column names. These files go to `data/clean/for_charts/`:

- `sponsors.csv`: the clean dataset with readable column names. Use it for filters and slicers (organization, region, year, industry).
- `industry_by_year.csv`: number of sponsor slots and share for each year × industry, ready for a stacked bar chart.
- `industry_by_region_year.csv`: the same counts split by region, for the crypto and betting line chart.

In [18]:
CHARTS = CLEAN / "for_charts"
os.makedirs(CHARTS, exist_ok=True)

INDUSTRY_LABELS = {
    "crypto_web3": "Crypto / Web3", "betting": "Betting & prediction markets",
    "energy_drink_food": "Energy drinks & food", "pc_hardware_peripherals": "PC hardware & peripherals",
    "telecom_tech": "Telecom & tech", "apparel": "Apparel", "automotive": "Automotive",
    "finance": "Finance", "other": "Other", "uncoded": "Not yet coded",
}

charts = tidy.copy()
charts["industry_label"] = charts["industry"].map(INDUSTRY_LABELS).fillna(charts["industry"])
charts = charts.rename(columns={
    "org": "Organization", "region": "Region", "year": "Year", "period": "Period",
    "capture_date": "Capture date", "sponsor_name": "Sponsor", "sponsor_domain": "Sponsor website",
    "industry": "Industry code", "industry_label": "Industry", "snapshot_url": "Archived page"})
charts = charts[["Organization", "Region", "Year", "Period", "Capture date", "Sponsor",
                 "Sponsor website", "Industry", "Industry code", "Archived page"]]
charts.to_csv(CHARTS / "sponsors.csv", index=False, encoding="utf-8-sig")

by_year = charts.groupby(["Year", "Industry"]).size().rename("Sponsor slots").reset_index()
by_year["Share of year"] = (by_year["Sponsor slots"]
                            / by_year.groupby("Year")["Sponsor slots"].transform("sum")).round(4)
by_year.to_csv(CHARTS / "industry_by_year.csv", index=False, encoding="utf-8-sig")

by_region = charts.groupby(["Region", "Year", "Industry"]).size().rename("Sponsor slots").reset_index()
by_region.to_csv(CHARTS / "industry_by_region_year.csv", index=False, encoding="utf-8-sig")

print("Saved:", *sorted(p.name for p in CHARTS.iterdir()), sep="\n  ")
by_year.pivot(index="Year", columns="Industry", values="Sponsor slots").fillna(0).astype(int)

Saved:
  industry_by_region_year.csv
  industry_by_year.csv
  sponsors.csv


Industry,Not yet coded
Year,
2019,70
2020,80
2021,111
2022,205
2023,175
2024,124
2025,121
2026,121
